In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [2]:
data = sns.load_dataset('titanic')
data.isnull().sum().sort_values(ascending=False)

deck           688
age            177
embarked         2
embark_town      2
sex              0
pclass           0
survived         0
fare             0
parch            0
sibsp            0
class            0
adult_male       0
who              0
alive            0
alone            0
dtype: int64

In [3]:
data.shape

(891, 15)

In [4]:
data.drop('deck',axis=1, inplace=True)
data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


In [5]:
# encoding the columns
encoded_columns = ['sex','embarked','class','who','adult_male','embark_town','alive']

label_encoders = {}
for col in encoded_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le
data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,1,22.0,1,0,7.2500,2,2,1,1,2,0,False
1,1,1,0,38.0,1,0,71.2833,0,0,2,0,0,1,False
2,1,3,0,26.0,0,0,7.9250,2,2,2,0,2,1,True
3,1,1,0,35.0,1,0,53.1000,2,0,2,0,2,1,False
4,0,3,1,35.0,0,0,8.0500,2,2,1,1,2,0,True


In [6]:
print(label_encoders)

{'sex': LabelEncoder(), 'embarked': LabelEncoder(), 'class': LabelEncoder(), 'who': LabelEncoder(), 'adult_male': LabelEncoder(), 'embark_town': LabelEncoder(), 'alive': LabelEncoder()}


In [7]:
df_with_missing = data[data['age'].isna()]
df_without_missing = data.dropna()
df_without_missing.shape

(714, 14)

In [8]:
x = df_without_missing.drop('age',axis=1)
y = df_without_missing['age']

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3,random_state=42)

model = RandomForestRegressor()
model.fit(x_train,y_train)
y_pred = model.predict(x_test)

print('r2 score :',r2_score(y_test,y_pred))
print('sqrt of mean squre error :',np.sqrt(mean_squared_error(y_test,y_pred)))

r2 score : 0.3716583011656671
sqrt of mean squre error : 10.758573447868095


In [9]:
# filling nan with random forest

x_missing = df_with_missing.drop('age',axis=1)
pred_age = model.predict(x_missing)

# df_with_missing['age'] = pred_age
df_with_missing.loc[df_with_missing['age'].isna(), 'age'] = pred_age
df_with_missing.isnull().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [10]:
df_with_missing.shape

(177, 14)

In [11]:
clean_data = pd.concat([df_with_missing, df_without_missing], axis=0)

In [12]:
clean_data.shape

(891, 14)

In [13]:
clean_data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
5,0,3,1,32.458083,0,0,8.4583,1,2,1,1,1,0,True
17,1,2,1,35.033134,0,0,13.0000,2,1,1,1,2,1,True
19,1,3,0,20.025250,0,0,7.2250,0,2,2,0,0,1,True
26,0,3,1,36.780000,0,0,7.2250,0,2,1,1,0,0,True
28,1,3,0,21.336000,0,0,7.8792,1,2,2,0,1,1,True


In [14]:
clean_data = data.copy()  # make a copy so you don’t overwrite

for col in encoded_columns:
    le = label_encoders[col]   # reuse the fitted encoder
    clean_data[col] = le.inverse_transform(data[col])

clean_data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True
